In [1]:
import pandas as pd

In [3]:
# Define the filename in your Colab environment
EPOCH_DATA_PATH = "/content/epoch_ai_compute.csv"

print(f"Loading Epoch AI compute data from: {EPOCH_DATA_PATH}")

try:
    df_compute = pd.read_csv(EPOCH_DATA_PATH)
    print("✅ Data loaded successfully.")

    # --- Initial Inspection ---
    print("\n--- Raw Data Info ---")
    # This will show us column names, data types, and non-null counts
    df_compute.info()

    print("\n--- First 5 Rows of Raw Data ---")
    print(df_compute[['Model', 'Publication date', 'Training compute (FLOP)']].head())

except FileNotFoundError:
    print(f"\n❌ ERROR: File '{EPOCH_DATA_PATH}' not found.")
    print("Please make sure you have uploaded the file to this Colab session.")

Loading Epoch AI compute data from: /content/epoch_ai_compute.csv
✅ Data loaded successfully.

--- Raw Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 942 entries, 0 to 941
Data columns (total 46 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Model                              942 non-null    object 
 1   Organization                       924 non-null    object 
 2   Publication date                   942 non-null    object 
 3   Domain                             941 non-null    object 
 4   Task                               935 non-null    object 
 5   Parameters                         662 non-null    float64
 6   Parameters notes                   624 non-null    object 
 7   Training compute (FLOP)            507 non-null    float64
 8   Training compute notes             582 non-null    object 
 9   Training dataset                   586 non-null    object 
 10  Train

In [4]:
# --- 2. Clean, Transform, and Aggregate ---

# Create a working copy
df_cleaned = df_compute.copy()

# --- A. Standardize Date -> Year ---
# Convert 'Publication date' to datetime objects. 'coerce' will turn any errors into NaT (Not a Time)
df_cleaned['publication_date'] = pd.to_datetime(df_cleaned['Publication date'], errors='coerce')
# Extract the year into a new 'year' column
df_cleaned['year'] = df_cleaned['publication_date'].dt.year
print("Extracted 'year' from 'Publication date'.")

# --- B. Standardize Compute Column ---
# The column is 'Training compute (FLOP)'. We need to ensure it's a numeric type.
df_cleaned['compute_flops'] = pd.to_numeric(df_cleaned['Training compute (FLOP)'], errors='coerce')
print("Converted 'Training compute (FLOP)' to a numeric format.")

# --- C. Handle Missing Data ---
# For our analysis, we can only use rows that have both a valid year and a valid compute value.
# Let's drop any rows where these critical values are missing.
original_rows = len(df_cleaned)
df_cleaned.dropna(subset=['year', 'compute_flops'], inplace=True)
print(f"Dropped {original_rows - len(df_cleaned)} rows with missing year or compute data.")

# --- D. Final Aggregation ---
# This is the most important step. We group all entries by year and SUM their compute values.
# This gives us one single data point for total compute per year.
df_aggregated = df_cleaned.groupby('year')['compute_flops'].sum().reset_index()

# Rename columns for clarity in the final output
df_aggregated.rename(columns={'compute_flops': 'total_compute_flops'}, inplace=True)

# Ensure the 'year' column is an integer
df_aggregated['year'] = df_aggregated['year'].astype(int)

print("\n✅ Data has been cleaned and aggregated by year.")

Extracted 'year' from 'Publication date'.
Converted 'Training compute (FLOP)' to a numeric format.
Dropped 435 rows with missing year or compute data.

✅ Data has been cleaned and aggregated by year.


In [5]:
# --- 3. Verify and Save Cleaned Data ---
from google.colab import files

print("--- Final Cleaned & Aggregated Data ---")
print(df_aggregated.info())

print("\n--- Data Preview ---")
# Print the entire final dataframe to see the full trend
print(df_aggregated)

# --- Save the final result to a new CSV file ---
output_filename = 'epoch_ai_compute_cleaned.csv'
df_aggregated.to_csv(output_filename, index=False)

print(f"\n✅ Successfully saved the cleaned data to '{output_filename}'.")

# --- Download the file to your computer ---
files.download(output_filename)
print(f"✅ The file '{output_filename}' has been downloaded to your computer.")

--- Final Cleaned & Aggregated Data ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   year                 50 non-null     int64  
 1   total_compute_flops  50 non-null     float64
dtypes: float64(1), int64(1)
memory usage: 932.0 bytes
None

--- Data Preview ---
    year  total_compute_flops
0   1950         4.000000e+01
1   1957         6.948949e+05
2   1959         1.028400e+09
3   1960         7.200066e+08
4   1962         1.559250e+06
5   1963         2.250000e+07
6   1965         1.080000e+06
7   1966         1.059171e+08
8   1975         5.184000e+06
9   1980         2.737382e+08
10  1983         3.240000e+08
11  1986         1.062720e+09
12  1987         7.402501e+10
13  1988         2.964250e+08
14  1989         1.712264e+12
15  1990         1.176465e+11
16  1991         7.547400e+10
17  1992         1.823221e+13
18  19

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ The file 'epoch_ai_compute_cleaned.csv' has been downloaded to your computer.
